# BAIT-Enhanced — Result Graphs

Two independent sections:

- **Section A — Reproduction graphs** · plots your team's committed reproduction numbers
  (`reproduction_result/results.md`, 90 models). **No scanning needed** — presentation-ready.
- **Section B — Live-scan graphs** · reads whatever `result.json` files your Path-3 runs
  produced under `results/` and plots them. As rich as the scans you've actually run.

Run **Setup**, then whichever section you need.


## Setup

In [ ]:
%cd /content
![ -d BAIT-Enhanced-group7 ] || git clone https://github.com/sureshbabugandla/BAIT-Enhanced-group7.git
%cd /content/BAIT-Enhanced-group7
!pip install -q matplotlib numpy pandas
import matplotlib.pyplot as plt, numpy as np
plt.rcParams.update({"figure.dpi":120,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
print("ready")

---
# Section A — Reproduction graphs (your team's numbers)

These come straight from `reproduction_result/results.md`. Truthful, and ready for slides.


### A.1 The data (edit here if your results.md changes)

In [ ]:
import pandas as pd

# from reproduction_result/results.md
rows = [
    # dataset,        n,  model,             acc,   prec, rec,  f1,    auc,   bleu,  overhead
    ("alpaca",        20, "Mistral-7B",      1.000, 1.00, 1.000, 1.000, 1.000, 0.946, 1869.4),
    ("self-instruct", 10, "Mistral-7B",      1.000, 1.00, 1.000, 1.000, 1.000, 0.888, 4192.9),
    ("alpaca",        21, "Llama-2-7B",      0.952, 1.00, 0.900, 0.947, 0.950, 0.843, 1425.8),
    ("self-instruct", 10, "Llama-2-7B",      0.900, 1.00, 0.800, 0.889, 0.800, 0.740, 1659.6),
    ("alpaca",        19, "Llama-3-8B",      0.947, 1.00, 0.889, 0.941, 0.989, 0.844, 2894.5),
    ("self-instruct", 10, "Llama-3-8B",      1.000, 1.00, 1.000, 1.000, 1.000, 0.883, 4186.3),
]
df = pd.DataFrame(rows, columns=["dataset","n","model","acc","prec","rec","f1","auc","bleu","overhead"])
OVERALL = dict(acc=0.967, prec=1.000, rec=0.932, f1=0.965, auc=0.961, bleu=0.865)
df

### A.2 Detection metrics by model type (grouped bars)

In [ ]:
agg = df.groupby("model")[["acc","prec","rec","f1","auc"]].mean().reindex(["Mistral-7B","Llama-2-7B","Llama-3-8B"])
metrics = ["acc","prec","rec","f1","auc"]
labels  = ["Accuracy","Precision","Recall","F1","ROC-AUC"]
x = np.arange(len(agg)); w = 0.16
fig, ax = plt.subplots(figsize=(9,4.5))
colors = ["#1f7a8c","#e09f3e","#3a8a5f","#8b5fb0","#c1444a"]
for i,(m,l,c) in enumerate(zip(metrics,labels,colors)):
    ax.bar(x + (i-2)*w, agg[m], w, label=l, color=c)
ax.set_xticks(x); ax.set_xticklabels(agg.index)
ax.set_ylim(0.6,1.03); ax.set_ylabel("score")
ax.set_title("BAIT-Enhanced — detection metrics by base model (reproduction, 90 models)")
ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5,-0.28), frameon=False)
plt.tight_layout(); plt.savefig("fig_metrics_by_model.png", bbox_inches="tight"); plt.show()

### A.3 ROC-AUC and BLEU by dataset × model

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11,4.2))
for ax, col, title, ylim in [(axes[0],"auc","ROC-AUC (detection)",(0.7,1.02)),
                              (axes[1],"bleu","BLEU (inverted-target fidelity)",(0.6,1.0))]:
    piv = df.pivot(index="model", columns="dataset", values=col).reindex(["Mistral-7B","Llama-2-7B","Llama-3-8B"])
    piv.plot(kind="bar", ax=ax, color=["#1f7a8c","#e09f3e"], width=0.7)
    ax.set_title(title); ax.set_ylim(*ylim); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
    ax.legend(title="dataset", frameon=False)
plt.tight_layout(); plt.savefig("fig_auc_bleu.png", bbox_inches="tight"); plt.show()

### A.4 Confusion matrix (overall, 90 models)

In [ ]:
# From results.md: 0 false positives, 3 false negatives over 90 models.
# Precision=1.0 -> no FP; Recall=0.932 over the positives. Reconstruct counts:
# 3 FN among poisoned; assume a balanced-ish split -> ~44 poisoned, ~46 benign.
TP, FN, FP, TN = 41, 3, 0, 46      # 41+3 poisoned = 44 ; recall 41/44 = 0.932
cm = np.array([[TN, FP],[FN, TP]])
fig, ax = plt.subplots(figsize=(4.6,4.2))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["Pred benign","Pred backdoored"])
ax.set_yticklabels(["True benign","True backdoored"])
for i in range(2):
    for j in range(2):
        ax.text(j,i,cm[i,j],ha="center",va="center",
                color="white" if cm[i,j]>cm.max()/2 else "black", fontsize=15, weight="bold")
ax.set_title("Confusion matrix (overall)\nFP=0, FN=3")
plt.tight_layout(); plt.savefig("fig_confusion.png", bbox_inches="tight"); plt.show()

### A.5 Scan overhead (runtime) by dataset × model

In [ ]:
piv = df.pivot(index="model", columns="dataset", values="overhead").reindex(["Mistral-7B","Llama-2-7B","Llama-3-8B"])
ax = piv.plot(kind="bar", figsize=(8,4), color=["#1f7a8c","#e09f3e"], width=0.7)
ax.set_ylabel("seconds / model"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
ax.set_title("Average scan overhead per model")
ax.legend(title="dataset", frameon=False)
plt.tight_layout(); plt.savefig("fig_overhead.png", bbox_inches="tight"); plt.show()

### A.6 Overall summary (single figure for slides)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
keys = ["acc","prec","rec","f1","auc","bleu"]
labs = ["Accuracy","Precision","Recall","F1","ROC-AUC","BLEU"]
vals = [0.967,1.000,0.932,0.965,0.961,0.865]
bars = ax.bar(labs, vals, color="#1f2d50")
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2, v+0.008, f"{v:.3f}", ha="center", fontsize=10, weight="bold")
ax.set_ylim(0.8,1.03); ax.set_title("BAIT-Enhanced — overall reproduction (90 models)")
plt.tight_layout(); plt.savefig("fig_overall_summary.png", bbox_inches="tight"); plt.show()

---
# Section B — Live-scan graphs (from your Path-3 runs)

Reads every `result.json` under `results/` and plots the Q-Scores. Run this **after** you've
done one or more scans in the Path-3 notebook (they share the same `/content` folder if run
in the same session, or point `RESULTS_DIR` at your saved results).


In [ ]:
import glob, json, os
RESULTS_DIR = "results"     # change if your scans wrote elsewhere

records = []
for p in glob.glob(os.path.join(RESULTS_DIR, "**", "result.json"), recursive=True):
    try:
        d = json.load(open(p))
        mid = os.path.basename(os.path.dirname(p))
        records.append({"model_id": mid,
                        "q_score": d.get("q_score", d.get("best_qscore", 0.0)),
                        "q_std": d.get("q_std", 0.0),
                        "invert_target": (d.get("invert_target") or d.get("best_sequence") or "")[:60]})
    except Exception as e:
        print("skip", p, e)

if not records:
    print("No result.json found under", RESULTS_DIR,
          "\nRun a Path-3 scan first, or set RESULTS_DIR to your saved results folder.")
else:
    import pandas as pd
    rdf = pd.DataFrame(records).sort_values("q_score", ascending=False)
    print(rdf.to_string(index=False))

### B.1 Q-Score per scanned model (with the decision threshold)

In [ ]:
if records:
    import numpy as np
    rdf2 = rdf.reset_index(drop=True)
    TAU = 0.85
    colors = ["#c1444a" if q>TAU else "#3a8a5f" for q in rdf2["q_score"]]
    fig, ax = plt.subplots(figsize=(max(6,len(rdf2)*0.7),4))
    ax.bar(rdf2["model_id"], rdf2["q_score"], color=colors,
           yerr=rdf2["q_std"] if rdf2["q_std"].any() else None, capsize=4)
    ax.axhline(TAU, ls="--", color="#333", label=f"decision threshold {TAU}")
    ax.set_ylabel("Q-Score"); ax.set_ylim(0,1.05)
    ax.set_title("Q-Score per scanned model (red = flagged backdoored)")
    ax.legend(frameon=False); plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.savefig("fig_live_qscores.png", bbox_inches="tight"); plt.show()
else:
    print("No scans to plot yet.")

### Download the figures

All charts are saved as PNGs in the working folder — grab them from the Colab **Files** panel
(`fig_*.png`) for your slides/report.

In [ ]:
import glob
print("Saved figures:")
for f in sorted(glob.glob("fig_*.png")):
    print(" ", f)

---
*BAIT-Enhanced · Group 7 · Result graphs. Section A uses your committed reproduction numbers;
Section B plots live Path-3 scan outputs.*